In [1]:
import os
from pathlib import Path
import requests
from urllib.request import build_opener
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

In [2]:
# download TraCE21k data

txt_path = Path.cwd() / 'data' / 'TraCE-21k' / 'download_paths.txt'
output_path = Path.cwd() / 'data' / 'TraCE-21k' / 'data'
os.makedirs(output_path, exist_ok=True)

opener = build_opener()

with open(txt_path, "r") as f:
    full_filelist = [line.strip() for line in f if line.strip()]

keep = {"TSA", "TSOI"}
filelist = [f for f in full_filelist if any(f".{var}." in f for var in keep)]

for file in tqdm(filelist):
    ofile = os.path.basename(file)
    dest = output_path / ofile

    if dest.exists():
        #tqdm.write(f"Skipping {ofile} (already exists)")
        continue

    #tqdm.write(f"Downloading {ofile} ...")
    infile = opener.open(file)
    with open(dest, "wb") as outfile:
        outfile.write(infile.read())
    #tqdm.write(f"Done: {ofile}")

  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# download CHELSA-TraCE21k-centennial data

txt_path = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'envidatS3paths.txt'
output_path = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

with open(txt_path) as f:
    urls = [line.strip() for line in f if line.strip()]

#keep = {"tasmax"}
keep = {"tasmin"}
urls = [u for u in urls if any(f"_{v}_" in u for v in keep)]
urls = urls[1000:1500]

def download(url):
    dest = output_path / os.path.basename(url)
    if dest.exists():
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as out:
            for chunk in r.iter_content(chunk_size=1 << 20):
                out.write(chunk)
    tmp.rename(dest)

with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(download, u) for u in urls]
    for fut in tqdm(as_completed(futures), total=len(urls)):
        try:
            fut.result()
        except Exception as e:
            tqdm.write(f"Failed: {e}")

  0%|          | 0/500 [00:00<?, ?it/s]

In [4]:
# convert CHELSA-TraCE21k-centennial data to netcdf

In [5]:
# cdo make decadal averages from monthly data

In [6]:
# compute mean from tasmax and tasmax

In [7]:
# download beyer2020 data

ARTICLE_ID = 12293345
#VERSION = 3  # Beyer2020
VERSION = 4  # Beyer2021

API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{VERSION}/files"

# download LateQuaternary_Environment data
output_path = Path.cwd() / 'data' / 'Beyer2020' / 'data'
os.makedirs(output_path, exist_ok=True)

response = requests.get(API_URL)
response.raise_for_status()
filelist = response.json()  # list of dicts with 'name' and 'download_url'

for file in tqdm(filelist):
    ofile = file['name']
    dest = output_path / ofile
    if dest.exists():
        continue
    r = requests.get(file['download_url'], stream=True)
    r.raise_for_status()
    with open(dest, 'wb') as outfile:
        for chunk in r.iter_content(chunk_size=8192):
            outfile.write(chunk)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# download PalMod2 data

import pexpect
import getpass

# config
username  = "lgiersch@gfz.de"

jblob_path = Path.cwd() / 'data' / 'PalMod2' / 'jblob-4.1' / 'jblob.sh'

output_path = Path.cwd() / 'data' / 'PalMod2' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "tas": "PMMXMCRTDGr111Amtasgn30201",  # near-surface air temperature, monthly
    "tsl": "PMMXMCRTDGr111Lmtslgn30201",  # soil temperature by layer, monthly
}

# login to wdcc
password = getpass.getpass("Enter WDCC password: ")

child = pexpect.spawn(f"bash {jblob_path} --login", encoding="utf-8", timeout=30)
child.logfile = None  # set to sys.stdout to debug

child.expect("Account type.*:")
child.sendline("WDCC")

child.expect("Username.*:")
child.sendline(username)

child.expect("Password.*:")
child.sendline(password)

child.expect(["Login successful", pexpect.EOF])
print("Login:", child.before + child.after)
child.close()

# download
for name, dataset_id in DATASETS.items():
    # skip if already downloaded
    if list(output_path.glob(f"*{dataset_id}*")):
        print(f"Skipping {name} (already exists)")
        continue

    print(f"\nDownloading {name} ({dataset_id}) ...")
    child = pexpect.spawn(
        f"bash {jblob_path} --dataset {dataset_id} --dir {output_path}",
        encoding="utf-8", timeout=None
    )
    child.logfile = None
    child.expect(pexpect.EOF)
    print(f"Done: {name}")

print("\nAll downloads complete.")